# LEGA-C DR3 against DR2

- Cross-match, spectra, selection and paired fits, DR3 vs DR2.

In [1]:
from pathlib import Path
import json
import sys
import h5py
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table
from numpy.polynomial import legendre
from specutils.utils.wcs_utils import air_to_vac

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/experiment.py').exists())
sys.path.insert(0, str(ROOT))
from scripts.experiment import validate_result
from ceridwen.observation.absorption_features import ABSORPTION_FEATURES, feature_windows

OUT = ROOT / 'results/legac-dr3-2026-11-11'
RAW = ROOT / 'data/raw'
TEST_SET = [t['spect_id'] for t in json.loads((ROOT / 'results/quiescent-test-set/targets.json').read_text())['targets']]
DR2_135 = [t['spect_id'] for t in json.loads((ROOT / 'results/dr2-refit-2026-10-12/targets.json').read_text())['targets']]
DR2_187 = [t['spect_id'] for t in json.loads((ROOT / 'results/dr2-quiescent-new-defaults/targets.json').read_text())['targets']]
WINDOW = (4074.5, 4666.1)  # chronometer arm (i) rest_range [vacuum A]
TELLURIC_AIR = (7590.0, 7660.0)  # notebook telluric_air mask [observed air A]
# DR2 telluric-corrected ranges [observed air A], Straatman et al. 2018 Sect. 2 step 5
DR2_TELLURIC = [(6865, 6960), (7160, 7330), (7580, 7710), (8070, 8400), (8900, 9200)]
assert len(TEST_SET) == 10 and len(DR2_135) == 135 and len(DR2_187) == 187

- Catalogue rows by `SPECT_ID`; DR3 renames `z`, `_err` to `Z`, `_ERR`.

In [2]:
def text(value):
    return (value.decode() if isinstance(value, bytes) else str(value)).strip()


def catalogue(release):
    path = RAW / 'legac_dr2/legaCdr2.fits.gz' if release == 'dr2' else RAW / 'legac_dr3/legac_DR3.fits'
    table = Table.read(path, hdu=1).to_pandas()
    if release == 'dr2':
        table = table.rename(columns=lambda c: 'Z' if c == 'z' else c.removesuffix('_err') + '_ERR' if c.endswith('_err') else c)
    for column in ('SPECT_ID', 'Filename'):
        table[column] = table[column].map(text)
    return table


def spectrum(release, filename):
    with fits.open(RAW / f'legac_{release}/sp' / filename) as hdulist:
        data = hdulist[1].data
        return {k: np.asarray(data[k][0], float) for k in ('WAVE', 'FLUX', 'ERR', 'QUAL')}, dict(hdulist[0].header)


CAT = {release: catalogue(release) for release in ('dr2', 'dr3')}
BY_ID = {release: table.set_index('SPECT_ID') for release, table in CAT.items()}
{release: len(table) for release, table in CAT.items()}

{'dr2': 1988, 'dr3': 4081}

- DR2 135 and test set in DR3: same `OBJECT`, separation, Δz.

In [3]:
def crossmatch(ids):
    dr2, dr3 = BY_ID['dr2'].loc[ids], BY_ID['dr3'].reindex(ids)
    sep = SkyCoord(dr2.RAJ2000.to_numpy() * u.deg, dr2.DECJ2000.to_numpy() * u.deg).separation(
        SkyCoord(dr3.RAJ2000.to_numpy() * u.deg, dr3.DECJ2000.to_numpy() * u.deg)).arcsec
    return pd.DataFrame(dict(target=ids, in_dr3=dr3.OBJECT.notna().to_numpy(), same_object=(dr2.OBJECT == dr3.OBJECT).to_numpy(),
                             separation_arcsec=sep, z_dr2=dr2.Z.to_numpy(), z_dr3=dr3.Z.to_numpy(), dz=(dr3.Z - dr2.Z).to_numpy()))


match = pd.concat([crossmatch(DR2_135).assign(sample='dr2_135'), crossmatch(TEST_SET).assign(sample='test_set')])
match.to_csv(OUT / 'crossmatch.csv', index=False)
match.groupby('sample').agg(galaxies=('target', 'size'), in_dr3=('in_dr3', 'sum'), same_object=('same_object', 'sum'),
                            max_separation_arcsec=('separation_arcsec', 'max'), max_abs_dz=('dz', lambda d: d.abs().max()))

,galaxies,in_dr3,same_object,max_separation_arcsec,max_abs_dz
sample,,,,,
dr2_135,135,135,135,0.0,0.002900
test_set,10,10,10,0.0,0.000528


- Test-set spectra: grid, good pixels, flux and error ratios.
- Good pixel: `QUAL` 0, `ERR` > 0, finite flux, outside the A band.
- Smooth flux ratio: order-5 Legendre fit to DR3/DR2.
- `chi`: DR3 minus scaled DR2, over the combined error.

In [4]:
def good(s):
    return (s['QUAL'] == 0) & (s['ERR'] > 0) & np.isfinite(s['FLUX'])


def compare(target):
    (s2, h2), (s3, h3) = (spectrum(r, BY_ID[r].loc[target, 'Filename']) for r in ('dr2', 'dr3'))
    w = s2['WAVE']
    g2, g3 = good(s2), good(s3)
    both = g2 & g3 & ~((w >= TELLURIC_AIR[0]) & (w <= TELLURIC_AIR[1]))
    ratio = s3['FLUX'][both] / s2['FLUX'][both]
    wb = w[both]
    x = 2 * (wb - wb.min()) / (wb.max() - wb.min()) - 1
    coef = legendre.legfit(x, ratio, 5)
    curve = legendre.legval(np.linspace(-1, 1, 200), coef)
    at = legendre.legval(x, coef)
    chi = (s3['FLUX'][both] - at * s2['FLUX'][both]) / np.hypot(s3['ERR'][both], at * s2['ERR'][both])
    in_tell = np.any([(wb >= lo) & (wb <= hi) for lo, hi in DR2_TELLURIC], axis=0)
    err = s3['ERR'][both] / s2['ERR'][both]
    c2, c3 = BY_ID['dr2'].loc[target], BY_ID['dr3'].loc[target]
    return dict(
        target=target, grid_identical=bool(np.array_equal(s2['WAVE'], s3['WAVE'])),
        wavelmin_dr2=h2['WAVELMIN'] * 10, wavelmin_dr3=h3['WAVELMIN'] * 10,
        wavelmax_dr2=h2['WAVELMAX'] * 10, wavelmax_dr3=h3['WAVELMAX'] * 10,
        good_dr2=int(g2.sum()), good_dr3=int(g3.sum()), good_only_dr2=int((g2 & ~g3).sum()), good_only_dr3=int((~g2 & g3).sum()),
        flux_ratio_median=np.median(ratio), flux_ratio_blue=curve[0], flux_ratio_red=curve[-1],
        flux_ratio_peak_to_peak=(curve.max() - curve.min()) / np.median(ratio),
        chi_std=np.std(chi), chi_std_dr2_telluric=np.std(chi[in_tell]), chi_std_outside=np.std(chi[~in_tell]),
        err_ratio_median=np.median(err), err_ratio_over_flux_ratio=np.median(err / ratio),
        sn_ratio_median=np.median(ratio / err),
        sn_cat_dr2=c2.SN, sn_cat_dr3=c3.SN,
        sigma_star_dr2=c2.SIGMA_STARS_PRIME, sigma_star_dr3=c3.SIGMA_STARS_PRIME,
        sigma_star_err_dr2=c2.SIGMA_STARS_PRIME_ERR, sigma_star_err_dr3=c3.SIGMA_STARS_PRIME_ERR,
        flag_spec_dr3=int(c3.FLAG_SPEC), flag_morph_dr3=int(c3.FLAG_MORPH))


spectra = pd.DataFrame([compare(t) for t in TEST_SET])
spectra.to_csv(OUT / 'spectra-dr3-vs-dr2.csv', index=False)
spectra.round(3).set_index('target').T

target,M1_210210,M8_150848,M5_238314,M3_120308,M2_134021,M7_146213,M3_117010,M4_107370,M14_37219,M8_161113
grid_identical,True,True,True,True,True,True,True,True,True,True
wavelmin_dr2,6186.1,6740.5,6365.5,6721.9,6677.5,6585.1,5974.9,6830.5,6192.7,6239.5
wavelmin_dr3,6189.7,6744.1,6369.1,6725.5,6680.5,6587.5,5978.5,6833.5,6195.7,6242.5
wavelmax_dr2,8900.5,9229.9,8828.5,9203.5,9163.9,9066.1,8410.9,9324.1,8642.5,8694.1
wavelmax_dr3,8631.7,9225.1,8824.9,9198.7,9159.1,9062.5,8407.3,9320.5,8638.3,8689.3
good_dr2,4053,4099,4061,4064,4097,4074,3989,4056,4041,4042
good_dr3,4053,4119,4052,4045,3915,4056,3985,4076,4056,4044
good_only_dr2,13,10,31,38,197,54,26,27,7,21
good_only_dr3,13,30,22,19,15,36,22,47,22,23
flux_ratio_median,0.86,0.941,1.086,0.871,0.828,1.021,0.881,0.864,0.889,1.123


- DR2 cuts of the notebook, and their DR3 analogue.
- DR3 flags: `FLAG_PRIMARY` 1, `FLAG_SPEC` 0, `FLAG_MORPH` 0 (van der Wel et al. 2021 Sect. 4.2).
- DR2 `f_use` = primary, `f_spec`, `f_z`, `f_ppxf` all good (Straatman et al. 2018 Sect. 3.3).
- DR3 emission EWs are positive, DR2 negative: sign check below.

In [5]:
shared = BY_ID['dr2'].index.intersection(BY_ID['dr3'].index)
ew2, ew3 = BY_ID['dr2'].loc[shared, 'OII_3727_EW'], BY_ID['dr3'].loc[shared, 'OII_3727_EW']
ok = ew2.notna() & ew3.notna() & (ew2.abs() > 1)
assert np.median(ew3[ok] / ew2[ok]) < -0.9
PHOT_COLUMNS = ['Area', 'Sat', 'Cfl', 'Flag', 'NUVMag', 'RMag', 'JMag']


def parent(release):
    phot = Table.read(RAW / f'cosmos2015/cosmos2015_legac_{release}_photometry_1arcsec.fits').to_pandas().set_index('LEGAC_INDEX')
    return CAT[release].join(phot[PHOT_COLUMNS], how='inner')


def select(p, release):
    counts = {'catalogue x COSMOS2015': len(p)}
    if release == 'dr2':
        q = (p.f_use == 1) & (p.f_ppxf == 0) & (p.f_z == 0) & (p.f_int == 0)
    else:
        q = (p.FLAG_PRIMARY == 1) & (p.FLAG_SPEC == 0) & p.Z.notna() & p.SIGMA_STARS_PRIME.notna() & (p.FLAG_MORPH == 0)
    q &= (p.SN > 0) & (p.Z >= 0.6) & (p.Z < 1.0) & (p[['NUVMag', 'RMag', 'JMag']] > -40).all(axis=1)
    p = p[q]; counts['quality flags, z, S/N, NUV r J'] = len(p)
    nuv_r, r_j = p.NUVMag - p.RMag, p.RMag - p.JMag
    p = p[(nuv_r > 3 * r_j + 1) & (nuv_r > 3.1)]; counts['NUVrJ quiescent'] = len(p)
    ew = p.OII_3727_EW
    p = p[((ew > -5) if release == 'dr2' else (ew < 5)) | ew.isna()]; counts['[O II] EW below 5 A'] = len(p)
    detected = lambda line: (p[f'{line}_EW'] / p[f'{line}_EW_ERR']).abs() >= 3
    p = p[~(detected('OII_3727') | detected('OIII_5007'))]; counts['no 3 sigma [O II], [O III]'] = len(p)
    p = p[(p.Area == 0) & (p.Sat == 0) & (p.Cfl == 1) & (p.Flag == 0)]; counts['clean COSMOS2015 photometry'] = len(p)
    p = p.sort_values(['SN', 'SPECT_ID'], ascending=[False, True]).drop_duplicates('OBJECT', keep='first')
    counts['one spectrum per OBJECT'] = len(p)
    return p, counts


SELECTED, COUNTS = {}, {}
for release in ('dr2', 'dr3'):
    SELECTED[release], COUNTS[release] = select(parent(release), release)
assert sorted(SELECTED['dr2'].SPECT_ID) == sorted(DR2_187)

- Fit photometry rule: COSMOS2025 `warn-flag` 0, else Classic `FlagCOMBINED` 0.
- Window: valid rest-vacuum pixels must span 4074.5–4666.1 Å.

In [6]:
def fittable(ids, release):
    paths = {'cosmos2025': RAW / f'cosmos2025/cosmos2025_phot_legac_{release}_1arcsec.fits',
             'classic': RAW / f'cosmos2020/cosmos2020_classic_legac_{release}_1arcsec.fits'}
    rows = {}
    for name, path in paths.items():
        table = Table.read(path)
        rows[name] = dict(zip(np.char.strip(np.asarray(table['SPECT_ID']).astype(str)), table))
    out = {}
    for s in ids:
        if s in rows['cosmos2025']:
            out[s] = ('cosmos2025', int(rows['cosmos2025'][s]['warn-flag']) == 0)
        elif s in rows['classic']:
            out[s] = ('cosmos2020_classic', int(rows['classic'][s]['FlagCOMBINED']) == 0)
        else:
            out[s] = ('none', False)
    return out


def rest_span(release, filename, z):
    s, _ = spectrum(release, filename)
    ok = good(s) & ~((s['WAVE'] >= TELLURIC_AIR[0]) & (s['WAVE'] <= TELLURIC_AIR[1]))
    rest = air_to_vac(s['WAVE'][ok] * u.AA).to_value(u.AA) / (1 + z)
    return rest.min(), rest.max()


samples = []
for release, sel in SELECTED.items():
    phot = fittable(sel.SPECT_ID, release)
    span = np.array([rest_span(release, f, z) for f, z in zip(sel.Filename, sel.Z)])
    samples.append(pd.DataFrame(dict(
        release=release, target=sel.SPECT_ID.to_numpy(), object_id=sel.OBJECT.to_numpy(), z=sel.Z.to_numpy(), sn=sel.SN.to_numpy(),
        photometry=[phot[s][0] for s in sel.SPECT_ID], fittable=[phot[s][1] for s in sel.SPECT_ID],
        rest_min=span[:, 0], rest_max=span[:, 1])))
samples = pd.concat(samples, ignore_index=True)
samples['covers_window'] = (samples.rest_min <= WINDOW[0]) & (samples.rest_max >= WINDOW[1])
dr2 = samples[samples.release == 'dr2'].set_index('target')
assert dr2.fittable.sum() == 180 and dr2.loc[DR2_135].fittable.all()
assert dr2.loc[DR2_135].covers_window.sum() == 110
samples['in_dr2_187'] = samples.target.isin(DR2_187)
samples['in_dr2_135'] = samples.target.isin(DR2_135)
samples.to_csv(OUT / 'selection-per-galaxy.csv', index=False)
for release in ('dr2', 'dr3'):
    s = samples[samples.release == release]
    COUNTS[release] |= {'fittable photometry': int(s.fittable.sum()), 'fittable, covers window': int((s.fittable & s.covers_window).sum())}
counts = pd.DataFrame(COUNTS)
counts.to_csv(OUT / 'selection-counts.csv')
counts

,dr2,dr3
catalogue x COSMOS2015,1982,4057
"quality flags, z, S/N, NUV r J",1328,2571
NUVrJ quiescent,609,1222
[O II] EW below 5 A,454,931
"no 3 sigma [O II], [O III]",349,581
clean COSMOS2015 photometry,194,329
one spectrum per OBJECT,187,302
fittable photometry,180,284
"fittable, covers window",144,208


- DR2 187 spectra not selected in DR3: reason for each.
- Same `OBJECT` selected with another DR3 spectrum: higher `SN`.

In [7]:
dr3_parent = parent('dr3').set_index('SPECT_ID')
lost = [s for s in DR2_187 if s not in set(SELECTED['dr3'].SPECT_ID)]
DR3_OBJECTS = set(SELECTED['dr3'].OBJECT)


def first_failure(row):
    if row.OBJECT in DR3_OBJECTS:
        return 'other spectrum of the OBJECT selected'
    if not ((row.FLAG_PRIMARY == 1) & (row.FLAG_SPEC == 0) & (row.FLAG_MORPH == 0)):
        return f'FLAG_PRIMARY {row.FLAG_PRIMARY}, FLAG_SPEC {row.FLAG_SPEC}, FLAG_MORPH {row.FLAG_MORPH}'
    oii = row.OII_3727_EW >= 5
    detected = abs(row.OII_3727_EW / row.OII_3727_EW_ERR) >= 3 or abs(row.OIII_5007_EW / row.OIII_5007_EW_ERR) >= 3
    return {(True, True): '[O II] EW and 3 sigma line', (True, False): '[O II] EW', (False, True): '3 sigma [O II] or [O III]'}.get(
        (bool(oii), bool(detected)), 'other')


lost_reasons = pd.Series([first_failure(dr3_parent.loc[s]) for s in lost], index=lost, name='first_dr3_failure')
lost_reasons.to_frame().assign(in_dr2_135=lambda d: d.index.isin(DR2_135)).to_csv(OUT / 'dr2-only-galaxies.csv')
lost_reasons.value_counts()

first_dr3_failure
3 sigma [O II] or [O III]                    47
[O II] EW and 3 sigma line                    8
other spectrum of the OBJECT selected         7
FLAG_PRIMARY 1, FLAG_SPEC 0, FLAG_MORPH 1     3
Name: count, dtype: int64

- Fits: DR3 `dr3_full`, `dr3_window`; DR2 references below.
- Full range: test-set baseline (`afe-ladder-floor-2026-11-10`, `free_afe`).
- Window: chronometer arm (i), 8 of 10 galaxies.

In [8]:
RUN = OUT / 'test-set/run/fits'
DR2_FULL = ROOT / 'results/afe-ladder-floor-2026-11-10/run/fits/free_afe'
DR2_WINDOW = ROOT / 'results/chronometer-135-2026-11-10/baseline-w110/run/fits/baseline_w110'
OBJECT = {s: int(BY_ID['dr3'].loc[s, 'OBJECT']) for s in TEST_SET}
FITS = {('full', 'dr3'): RUN / 'dr3_full', ('full', 'dr2'): DR2_FULL,
        ('window', 'dr3'): RUN / 'dr3_window', ('window', 'dr2'): DR2_WINDOW}
PARAMETERS = {'age_gyr': 'mass-weighted age [Gyr]', 'afe': 'afe', 'feh': 'logzsol', 'tau_dust': 'diffuse_tau_noll',
              'floor_pct': 'calibration floor [%]', 'sigma_smooth': 'sigma_smooth'}


def fit_dir(arm, release, target):
    return FITS[arm, release] / f'{OBJECT[target]}-{target}'


def fit_row(arm, release, target):
    directory = fit_dir(arm, release, target)
    row = dict(arm=arm, release=release, target=target, path=str(directory.relative_to(ROOT)))
    if not directory.exists():
        return row | dict(valid=False)
    validate_result(directory)
    source = json.loads((directory.parents[2] / 'manifest.json').read_text())['source']
    row |= dict(valid=True, astro_commit=source['commit'][:8], ceridwen_commit=source['submodules']['ceridwen'][:8])
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        names = f['summary/parameter'].asstr()[:]
        q = {k: dict(zip(names, f[f'summary/{k}'][:])) for k in ('q16', 'q50', 'q84')}
        row |= dict(z=float(f.attrs['redshift']), seed=int(f.attrs['random_seed']), photometry=f.attrs['photometry_source'],
                    sigma_star_prior=float(f.attrs['sigma_star_kms']), sigma_star_prior_err=float(f.attrs['sigma_star_err']),
                    fitted_pixels=int(f['spectrum/mask'][:].sum()),
                    photometry_pull_rms=float(np.sqrt(np.mean(f['photometry/pull'][:][f['photometry/mask'][:]] ** 2))))
    for label, name in PARAMETERS.items():
        row |= {f'{label}_{k}': float(q[k][name]) for k in q}
    return row


fits_table = pd.DataFrame([fit_row(arm, release, t) for arm, release in FITS for t in TEST_SET])
fits_table.to_csv(OUT / 'fits.csv', index=False)
valid = fits_table[fits_table.valid]
assert (valid.seed == 20260832).all()
valid.groupby(['arm', 'release']).agg(fits=('target', 'size'), astro=('astro_commit', 'unique'), ceridwen=('ceridwen_commit', 'unique'))

fits       astro    ceridwen
arm    release                              
full   dr2        10  [9b189a36]  [a0b1f6ed]
       dr3        10  [ed3d4f1b]  [13791e0e]
window dr2         8  [99c179ba]  [13791e0e]
       dr3        10  [ed3d4f1b]  [13791e0e]

- Paired change Δ = DR3 − DR2 median; σ = half the DR2 16–84 width.

In [9]:
pairs = []
for arm in ('full', 'window'):
    a = valid[(valid.arm == arm) & (valid.release == 'dr2')].set_index('target')
    b = valid[(valid.arm == arm) & (valid.release == 'dr3')].set_index('target')
    for target in a.index.intersection(b.index):
        row = dict(arm=arm, target=target, z=a.loc[target, 'z'], photometry_pull_rms_dr2=a.loc[target, 'photometry_pull_rms'],
                   photometry_pull_rms_dr3=b.loc[target, 'photometry_pull_rms'])
        for p in PARAMETERS:
            sigma2 = (a.loc[target, f'{p}_q84'] - a.loc[target, f'{p}_q16']) / 2
            sigma3 = (b.loc[target, f'{p}_q84'] - b.loc[target, f'{p}_q16']) / 2
            delta = b.loc[target, f'{p}_q50'] - a.loc[target, f'{p}_q50']
            row |= {f'{p}_dr2': a.loc[target, f'{p}_q50'], f'{p}_dr3': b.loc[target, f'{p}_q50'], f'{p}_delta': delta,
                    f'{p}_delta_over_sigma': delta / sigma2, f'{p}_width_ratio': sigma3 / sigma2}
        pairs.append(row)
pairs = pd.DataFrame(pairs)
pairs.to_csv(OUT / 'paired-changes-per-galaxy.csv', index=False)
summary = pd.DataFrame([dict(arm=arm, parameter=p, galaxies=len(g), median_delta=g[f'{p}_delta'].median(),
                             median_delta_over_sigma=g[f'{p}_delta_over_sigma'].median(),
                             max_abs_delta_over_sigma=g[f'{p}_delta_over_sigma'].abs().max(),
                             median_width_ratio=g[f'{p}_width_ratio'].median())
                        for arm, g in pairs.groupby('arm', sort=False) for p in PARAMETERS])
summary.to_csv(OUT / 'paired-changes.csv', index=False)
summary.round(3)

,arm,parameter,galaxies,median_delta,median_delta_over_sigma,max_abs_delta_over_sigma,median_width_ratio
0,full,age_gyr,10,-0.033,-0.083,3.532,1.020
1,full,afe,10,-0.010,-0.240,5.138,0.936
2,full,feh,10,0.177,3.581,8.127,0.965
3,full,tau_dust,10,-0.044,-1.217,2.209,1.033
4,full,floor_pct,10,0.118,0.662,20.522,1.010
5,full,sigma_smooth,10,-1.417,-0.313,3.383,0.906
6,window,age_gyr,8,0.102,0.207,2.676,1.051
7,window,afe,8,-0.009,-0.080,5.482,0.917
8,window,feh,8,0.143,2.217,5.107,0.922
9,window,tau_dust,8,-0.042,-0.966,3.152,1.012


- Calibration polynomial: DR3/DR2 posterior-median polynomial ratio, common fitted pixels.
- Compared with the smooth DR3/DR2 flux ratio on the same pixels.
- `track_rms`: rms of polynomial ratio over flux ratio, each normalised to its median.

In [10]:
def polynomial(directory):
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        return (f['spectrum/wavelength'][:].astype(float), f['spectrum/mask'][:].astype(bool),
                f['calibration/polynomial_q16'][:], f['calibration/polynomial_q50'][:], f['calibration/polynomial_q84'][:])


def flux_ratio_curve(target, wave):
    (s2, _), (s3, _) = (spectrum(r, BY_ID[r].loc[target, 'Filename']) for r in ('dr2', 'dr3'))
    w = s2['WAVE']
    both = good(s2) & good(s3) & ~((w >= TELLURIC_AIR[0]) & (w <= TELLURIC_AIR[1]))
    lo, hi = w[both].min(), w[both].max()
    coef = legendre.legfit(2 * (w[both] - lo) / (hi - lo) - 1, s3['FLUX'][both] / s2['FLUX'][both], 5)
    return legendre.legval(2 * (wave - lo) / (hi - lo) - 1, coef)


rows = []
for arm in ('full', 'window'):
    for target in pairs[pairs.arm == arm].target:
        w2, m2, lo2, p2, hi2 = polynomial(fit_dir(arm, 'dr2', target))
        w3, m3, _, p3, _ = polynomial(fit_dir(arm, 'dr3', target))
        assert np.array_equal(w2, w3)
        m = m2 & m3
        ratio = p3[m] / p2[m]
        flux = flux_ratio_curve(target, w2[m])
        rows.append(dict(arm=arm, target=target, pixels=int(m.sum()), poly_ratio_median=np.median(ratio),
                         poly_ratio_peak_to_peak=np.ptp(ratio) / np.median(ratio),
                         flux_ratio_peak_to_peak=np.ptp(flux) / np.median(flux),
                         track_rms=np.std(ratio / np.median(ratio) - flux / np.median(flux)),
                         dr2_poly_width=np.median((hi2[m] - lo2[m]) / 2 / p2[m])))
poly = pd.DataFrame(rows)
poly.to_csv(OUT / 'calibration-polynomial.csv', index=False)
poly.round(3)

,arm,target,pixels,poly_ratio_median,poly_ratio_peak_to_peak,flux_ratio_peak_to_peak,track_rms,dr2_poly_width
0,full,M1_210210,3599,0.855,0.133,0.115,0.011,0.013
1,full,M8_150848,3771,0.939,0.147,0.152,0.007,0.014
2,full,M5_238314,3566,1.071,0.096,0.144,0.021,0.013
3,full,M3_120308,3565,0.856,0.275,0.312,0.018,0.015
4,full,M2_134021,3434,0.824,0.182,0.206,0.014,0.015
5,full,M7_146213,3559,1.021,0.190,0.128,0.018,0.016
6,full,M3_117010,3509,0.887,0.127,0.141,0.014,0.015
7,full,M4_107370,3546,0.835,0.609,0.262,0.040,0.015
8,full,M14_37219,3570,0.882,0.173,0.204,0.013,0.013
9,full,M8_161113,3531,1.131,0.072,0.080,0.019,0.016


- Feature-band pull: Σ pull / √n over fitted pixels in each band.
- Bands with fitted pixels in both fits.

In [11]:
NAMES = [f.name for f in ABSORPTION_FEATURES]


def band_pulls(directory):
    with h5py.File(directory / 'ceridwen_derived_outputs.h5') as f:
        z = float(f.attrs['redshift'])
        wave = f['spectrum/wavelength'][:].astype(float)
        mask = f['spectrum/mask'][:].astype(bool)
        pull = f['spectrum/pull'][:]
    out = {}
    for name, (lo, hi) in zip(NAMES, feature_windows(NAMES, zred=z)):
        sel = mask & (wave >= lo) & (wave <= hi)
        if sel.any():
            out[name] = pull[sel].sum() / np.sqrt(sel.sum())
    return out


rows = []
for arm in ('full', 'window'):
    for target in pairs[pairs.arm == arm].target:
        b2, b3 = band_pulls(fit_dir(arm, 'dr2', target)), band_pulls(fit_dir(arm, 'dr3', target))
        rows += [dict(arm=arm, target=target, feature=n, pull_dr2=b2[n], pull_dr3=b3[n]) for n in NAMES if n in b2 and n in b3]
pulls = pd.DataFrame(rows).assign(delta=lambda d: d.pull_dr3 - d.pull_dr2)
pulls.to_csv(OUT / 'band-pulls-per-galaxy.csv', index=False)
band = pulls.groupby(['arm', 'feature'], sort=False).agg(
    galaxies=('delta', 'size'), median_dr2=('pull_dr2', 'median'), median_dr3=('pull_dr3', 'median'),
    median_delta=('delta', 'median'), max_abs_delta=('delta', lambda d: d.abs().max())).reset_index()
band.to_csv(OUT / 'band-pulls.csv', index=False)
band.round(2)

,arm,feature,galaxies,median_dr2,median_dr3,median_delta,max_abs_delta
0,full,H10,6,-1.40,-2.25,-0.53,1.79
1,full,H9,7,0.63,0.42,0.10,1.74
2,full,H8,8,1.51,1.13,-0.36,2.27
3,full,HdA,10,-0.60,-1.37,-0.90,2.11
4,full,HgA,10,1.53,1.54,0.45,3.17
5,full,Hbeta,9,0.57,-0.02,0.03,3.25
6,full,Ca4455,10,0.49,0.42,0.39,3.32
7,full,CN4150,10,-1.73,-2.46,-0.29,2.42
8,full,G4300,10,1.20,0.07,-1.79,3.68
9,full,C4668,9,-3.69,-3.95,-0.55,2.34
